# YouTube Music Playlist Export

This notebook takes a YouTube Music playlist (by name or list ID), downloads it via `ytmusicapi`, reverses the track order, queries Discogs for label info and other metadata, and exports a CSV ready to upload to zookeeper.stanford.edu.

## Target Schema
`artist,title,album,label,duration`

## Usage
```python
# By playlist name
export_playlist(playlist_name="DJ Stace - Next Show")

# By playlist ID
export_playlist(playlist_id="PLMduzOohHQLrQzt6w4ep6l2zAwcIzoZVF")
```

In [21]:
import csv
import os
import sys
import json
import requests
from typing import Optional, Dict, List, Any

# YTMusic API
try:
    from ytmusicapi import YTMusic
except ImportError:
    print("ytmusicapi not installed. Run: pip install ytmusicapi")
    sys.exit(1)

# Discogs API wrapper
# import discogs

## Configuration

Set your YTMusic authentication token and base URL for Discogs.

In [22]:
# Load browser.json for YTMusic OAuth authentication
BROWSER_JSON_PATH = "/Users/maples/Github/KZSU-Library-Show/browser.json"

with open(BROWSER_JSON_PATH, 'r') as f:
    browser_headers = json.load(f)

print("Browser headers loaded:")
print(f"  Origin: {browser_headers.get('origin', 'N/A')}")
print(f"  Authorization present: {'authorization' in browser_headers}")

# Metadata sources
DISCOGS_BASE = "https://api.discogs.com"
DISCOGS_USER_AGENT = "KZSU-Library-Show/1.0"
DISCOGS_TOKEN = os.getenv('DISCOGS_TOKEN', '')
CONSOLIDATED_CSV_PATH = "/Users/maples/Github/KZSU-Library-Show/notebooks/outputs/dj_stace_all_playlists_consolidated.csv"

# Output directory
OUTPUT_DIR = "/Users/maples/Github/KZSU-Library-Show/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

Browser headers loaded:
  Origin: https://music.youtube.com
  Authorization present: True


## Initialize YTMusic

Create the YTMusic instance with your authentication credentials.

In [23]:
def get_ytmusic_instance(browser_headers: Dict[str, str]) -> YTMusic:
    """
    Initialize and return a YTMusic instance using the browser auth JSON file.

    Recent ytmusicapi versions expect a path to the auth file; older versions accepted
    a raw browser_headers dict. Fall back to the legacy signature only if needed.

    Args:
        browser_headers: Dictionary containing browser request headers from browser.json

    Returns:
        Initialized YTMusic instance
    """
    auth_path = BROWSER_JSON_PATH
    if not os.path.exists(auth_path):
        raise FileNotFoundError(f"YTMusic auth file not found: {auth_path}")

    try:
        return YTMusic(str(auth_path))
    except TypeError:
        return YTMusic(browser_headers=browser_headers)


ytm = get_ytmusic_instance(browser_headers)
print("YTMusic instance initialized with browser auth file.")

YTMusic instance initialized with browser auth file.


## Fetch Playlist

Retrieve a playlist by name or list ID using ytmusicapi.

In [24]:
def fetch_playlist(
    playlist_name: Optional[str] = None,
    playlist_id: Optional[str] = None
) -> List[Dict[str, Any]]:
    """
    Fetch a YouTube Music playlist by name or list ID.

    Args:
        playlist_name: Name of the playlist to fetch
        playlist_id: ID of the playlist to fetch

    Returns:
        List of track dictionaries with metadata
    """
    if not playlist_name and not playlist_id:
        raise ValueError("Must provide either playlist_name or playlist_id")

    if playlist_id:
        print(f"Fetching playlist by ID: {playlist_id}")
        playlist = ytm.get_playlist(playlist_id)
    elif playlist_name:
        print(f"Fetching playlist by name: {playlist_name}")
        playlist = ytm.get_playlist(playlist_name)

    tracks = playlist.get('tracks', [])
    if not tracks:
        raise ValueError("Playlist is empty or not found")

    print(f"Retrieved {len(tracks)} tracks")
    return tracks

## Reverse Track Order

Reverse the order of tracks in the playlist.

In [25]:
def reverse_playlist(tracks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Reverse the order of tracks in a playlist.
    
    Args:
        tracks: List of track dictionaries
    
    Returns:
        Reversed list of track dictionaries
    """
    reversed_tracks = list(reversed(tracks))
    print(f"Reversed {len(reversed_tracks)} tracks")
    return reversed_tracks

## Discogs Lookup

Query Discogs for label info and other missing metadata for each track.

In [26]:
def _metadata_key(artist: str, title: str) -> tuple[str, str]:
    """Build a stable key for case-insensitive artist/title matching."""
    return artist.strip().casefold(), title.strip().casefold()


def load_consolidated_metadata(path: str) -> Dict[tuple[str, str], Dict[str, str]]:
    """Load the consolidated playlist metadata indexed by artist and title."""
    metadata = {}
    with open(path, newline='', encoding='utf-8') as csv_file:
        reader = csv.DictReader(csv_file)
        for row in reader:
            key = _metadata_key(row.get('artist', ''), row.get('title', ''))
            if not all(key):
                continue

            record = metadata.setdefault(key, {
                'album': '',
                'label': '',
                'duration': ''
            })
            for field in record:
                value = (row.get(field, '') or '').strip()
                if value and not record[field]:
                    record[field] = value

    return metadata


CONSOLIDATED_METADATA = load_consolidated_metadata(CONSOLIDATED_CSV_PATH)
print(f"Loaded {len(CONSOLIDATED_METADATA)} consolidated track records.")


def lookup_discogs_info(artist: str, title: str) -> Dict[str, str]:
    """Query Discogs for release metadata when the consolidated file lacks a label."""
    info = {
        'label': '',
        'album': '',
        'duration': ''
    }
    headers = {'User-Agent': DISCOGS_USER_AGENT}
    if DISCOGS_TOKEN:
        headers['Authorization'] = f'Discogs token={DISCOGS_TOKEN}'

    try:
        response = requests.get(
            f"{DISCOGS_BASE}/database/search",
            params={
                'artist': artist,
                'track': title,
                'type': 'release',
                'per_page': 10
            },
            headers=headers,
            timeout=30
        )
        response.raise_for_status()
        results = response.json().get('results', [])
        for result in results:
            labels = result.get('label', [])
            if labels:
                info['label'] = ', '.join(labels)
                break
    except Exception as e:
        print(f"Discogs lookup failed for {artist} - {title}: {e}")

    return info


def lookup_track_metadata(artist: str, title: str) -> Dict[str, str]:
    """Use consolidated metadata first and Discogs as the fallback source."""
    consolidated = CONSOLIDATED_METADATA.get(_metadata_key(artist, title), {})
    if consolidated.get('label'):
        return consolidated

    discogs_info = lookup_discogs_info(artist, title)
    return {
        'album': consolidated.get('album', '') or discogs_info.get('album', ''),
        'label': consolidated.get('label', '') or discogs_info.get('label', ''),
        'duration': consolidated.get('duration', '') or discogs_info.get('duration', '')
    }

Loaded 3184 consolidated track records.


## Format Output

Format the reversed playlist data into the target schema for zookeeper.stanford.edu.

In [27]:
def format_for_zookeeper(
    tracks: List[Dict[str, Any]]
) -> List[Dict[str, Any]]:
    """
    Format playlist tracks for zookeeper.stanford.edu upload.

    Args:
        tracks: List of reversed YTMusic track dictionaries

    Returns:
        List of formatted dictionaries matching target schema
    """
    formatted = []

    for track in tracks:
        artists = track.get('artists', [])
        artist = ', '.join(
            item.get('name', '') if isinstance(item, dict) else str(item)
            for item in artists
        ) if isinstance(artists, list) else str(artists or '')

        title = track.get('title', '')
        album_data = track.get('album', '')
        album = album_data.get('name', '') if isinstance(album_data, dict) else str(album_data or '')
        duration = track.get('duration', '')

        metadata = lookup_track_metadata(artist, title)

        formatted.append({
            'artist': artist,
            'title': title,
            'album': album or metadata.get('album', ''),
            'label': metadata.get('label', ''),
            'duration': duration or metadata.get('duration', '')
        })

    return formatted

## Export CSV

Export the formatted data to a CSV file ready for zookeeper.stanford.edu upload.

In [28]:
def export_playlist(
    playlist_name: Optional[str] = None,
    playlist_id: Optional[str] = None,
    output_filename: str = "exported_playlist.csv"
) -> str:
    """
    Main function to export a YouTube Music playlist.

    Args:
        playlist_name: Name of the playlist
        playlist_id: ID of the playlist
        output_filename: Output CSV filename

    Returns:
        Path to the exported CSV file
    """
    # Fetch playlist
    tracks = fetch_playlist(playlist_name=playlist_name, playlist_id=playlist_id)

    # Reverse order
    reversed_tracks = reverse_playlist(tracks)

    # Format for zookeeper
    formatted = format_for_zookeeper(reversed_tracks)

    # Export to CSV without a header row; quote every field.
    import csv
    import pandas as pd
    df = pd.DataFrame(formatted)
    output_path = os.path.join(OUTPUT_DIR, output_filename)
    df.to_csv(output_path, index=False, header=False, quoting=csv.QUOTE_ALL)

    print(f"Exported {len(df)} tracks to {output_path}")
    return output_path

## Example Usage

Run the export function with your playlist name or ID.

In [29]:
# Example: Export by playlist name
#output_path = export_playlist(playlist_name="DJ Stace - Next Show")

# Example: Export by playlist ID
output_path = export_playlist(playlist_id="PLMduzOohHQLrQzt6w4ep6l2zAwcIzoZVF")

print("Ready to export. Uncomment one of the examples above and run.")

Fetching playlist by ID: PLMduzOohHQLrQzt6w4ep6l2zAwcIzoZVF
Retrieved 50 tracks
Reversed 50 tracks
Exported 50 tracks to /Users/maples/Github/KZSU-Library-Show/outputs/exported_playlist.csv
Ready to export. Uncomment one of the examples above and run.


## Target Schema Reference

The exported CSV matches the schema of `dj_stace_all_playlists_consolidated.csv`:
- `artist`: Artist name
- `title`: Track title
- `album`: Album name (from Discogs or original)
- `label`: Record label (from Discogs)
- `duration`: Track duration